Preprocesarea setului de date

Conectarea la drive si incarcarea csv ului

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
file_path = '/content/drive/MyDrive/IAPD/Proiect_IAPD/dataset.csv'
df = pd.read_csv(file_path)

verificare tipuri de date si val lipsa

In [ ]:
df.info()

transformarea distantelor in kilometri

In [ ]:
import pandas as pd
import re

df_original = df.copy()

def convert_to_km(distance):
    match = re.search(r'(\d+(?:[.,]\d+)?)\s*(km|m)\b', str(distance).lower())

    if match:
        value = float(match.group(1).replace(',', '.'))
        unit = match.group(2)

        if unit == 'm':
            return value / 1000

        return value

    return None

df['distance_km'] = df['distance_from_centre'].apply(convert_to_km)
df[['distance_from_centre', 'distance_km']].head(10)

identificam cele mai mici distante pentru fiecare proprietate

In [ ]:
min_distances = df.groupby('property_id')['distance_km'].transform('min')
# Pastram doar distantele minime dintre duplicate
df_closest = df[df['distance_km'] == min_distances].copy()

In [ ]:
# Proprietati duplicate cu distanta minima egala
tie_counts = df_closest.groupby('property_id').size()

tied_properties = tie_counts[tie_counts > 1].index

df_ties = df_closest[
    df_closest['property_id'].isin(tied_properties)
].copy()

df_ties[[
    'property_id',
    'name',
    'city',
    'distance_from_centre',
    'price_per_night',
    'unit_description'
]].sort_values('property_id')

Obtinem df_filtered prin scoaterea lui df_ties din df_closest

In [ ]:
df_filtered = df_closest[
    ~df_closest['property_id'].isin(df_ties['property_id'])
].copy()

Revenim la df_ties: Luam cazul in care distanta minima e egala pentru orase diferite.

In [ ]:
mask = df_ties.groupby('property_id')['city'].transform('nunique') > 1

df_ties_different_city = df_ties[mask].copy()
df_ties_different_city[[
    'property_id',
    'name',
    'city',
    'distance_from_centre',
    'price_per_night',
    'unit_description'
]].sort_values(['property_id', 'city'])


Am decis eliminarea completa a acestor cazuri

In [ ]:
df_ties = df_ties[
    df_ties.groupby('property_id')['city'].transform('nunique') == 1
].copy()

Pentru cazurile in care se repeta aceeasi cazare in acelasi oras:
1. Aceeasi proprietate cu property_type diferit cu acelasi pret
2. Aceeasi proprietate cu tipuri de camere diferie

In [ ]:
# Proprietati duplicate cu acelasi pret dar property_type diferit
property_type_duplicates = df_ties[
    df_ties.duplicated(
        subset=['property_id', 'price_per_night'],
        keep=False
    )
].sort_values(['property_id', 'price_per_night', 'property_type'])

property_type_duplicates[[
    'property_id',
    'name',
    'city',
    'property_type',
    'price_per_night',
    'unit_description'
]]

Alegem aleatoriu una din inregistrarile care se repeta si stergem din df_ties.

In [ ]:
duplicate_mask = df_ties.duplicated(
    subset=['property_id', 'price_per_night'],
    keep=False
)

duplicates_clean = (
    df_ties[duplicate_mask]
    .groupby(['property_id', 'price_per_night'], group_keys=False)
    .sample(n=1, random_state=42)
)

df_ties = pd.concat([
    df_ties[~duplicate_mask],
    duplicates_clean
]).sort_index().copy()


Luam acum cazul 2: cand la aceeasi cazare poti inchiria tipuri diferite de camere/ toata proprietatea, avand si preturi diferite

Am ales proprietatea cu el mai mic pret, ca sa avem o singura inregistrare per cazare

In [ ]:
min_prices = df_ties.groupby('property_id')['price_per_night'].transform('min')

df_ties = df_ties[df_ties['price_per_night'] == min_prices].copy()

df_ties = (
    df_ties.groupby('property_id', group_keys=False)
    .sample(n=1, random_state=42)
    .copy()
)

Adaugam ce am obtinut in df_filtered

In [ ]:
# Adaugam df_ties in df_filtered
df_filtered = pd.concat([df_filtered, df_ties], ignore_index=True)

# Verificam rezultatul
print("Numar total de randuri:", len(df_filtered))
print("Numar de property_id unice:", df_filtered['property_id'].nunique())
print("Numar de nume unice:", df_filtered['name'].nunique())

print("\nExista property_id duplicate:", df_filtered['property_id'].duplicated().any())
print("Exista nume duplicate:", df_filtered['name'].duplicated().any())

In [ ]:
# Identificam toate cazarile cu nume identice
duplicate_names = df_filtered[
    df_filtered['name'].duplicated(keep=False)
].sort_values('name')

# Afisam toate coloanele si toate inregistrarile
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

display(duplicate_names)

Sterg randurile 510 si 33 pentru ca sunt aceeasi cazare

In [ ]:
df_filtered = df_filtered.drop(index=[510, 33]).copy()

Salvez ce a ramas:

In [ ]:
# Salvam datasetul filtrat
output_path = '/content/drive/MyDrive/IAPD/Proiect_IAPD/dataset_filtered.csv'

df_filtered.to_csv(output_path, index=False, encoding='utf-8-sig')

print("Numar de randuri:", len(df_filtered))
print("Numar de coloane:", len(df_filtered.columns))